In [2]:
import h5py

def explore_hdf5_file(file_path):
    """
    Open an HDF5 file and display its structure and fields.
    
    Args:
        file_path: Path to the HDF5 file
    """
    def print_structure(name, obj):
        """Recursively print the structure of HDF5 objects"""
        indent = "  " * name.count('/')
        if isinstance(obj, h5py.Group):
            print(f"{indent}{name}/ (Group)")
            # Print attributes if any
            if obj.attrs:
                for attr_name, attr_value in obj.attrs.items():
                    print(f"{indent}  @{attr_name}: {attr_value}")
        elif isinstance(obj, h5py.Dataset):
            print(f"{indent}{name} (Dataset): shape={obj.shape}, dtype={obj.dtype}")
            # Print attributes if any
            if obj.attrs:
                for attr_name, attr_value in obj.attrs.items():
                    print(f"{indent}  @{attr_name}: {attr_value}")
    
    try:
        with h5py.File(file_path, 'r') as f:
            print(f"HDF5 file: {file_path}")
            print("=" * 50)
            
            # Print root level attributes
            if f.attrs:
                print("Root attributes:")
                for attr_name, attr_value in f.attrs.items():
                    print(f"  @{attr_name}: {attr_value}")
                print()
            
            # Print structure
            print("File structure:")
            f.visititems(print_structure)
            
            # Show top-level keys
            print("\nTop-level keys:")
            for key in f.keys():
                print(f"  {key}")
                
    except Exception as e:
        print(f"Error opening HDF5 file: {e}")

# Usage example:
# explore_hdf5_file('/path/to/your/file.h5')

In [4]:
explore_hdf5_file('/Volumes/aind/scratch/andrew.shelton/NPUltra_data/raw_npultra_data/2024-05-14_714527/behavior/RFMapping_714527_20240514_104628.hdf5')


HDF5 file: /Volumes/aind/scratch/andrew.shelton/NPUltra_data/raw_npultra_data/2024-05-14_714527/behavior/RFMapping_714527_20240514_104628.hdf5
File structure:
acquisitionSignalLine (Dataset): shape=(2,), dtype=int32
amNoiseFreq (Dataset): shape=(5,), dtype=int32
behavNidaqDevice (Dataset): shape=(), dtype=object
behavNidaqDeviceSerialNum (Dataset): shape=(), dtype=int32
computerName (Dataset): shape=(), dtype=float64
configPath (Dataset): shape=(), dtype=float64
deltaWheelPos (Dataset): shape=(0,), dtype=float64
digitalSolenoidTrigger (Dataset): shape=(), dtype=bool
diodeBoxPosition (Dataset): shape=(2,), dtype=int32
diodeBoxSize (Dataset): shape=(), dtype=int32
drawDiodeBox (Dataset): shape=(), dtype=bool
frameIntervals (Dataset): shape=(71999,), dtype=float64
frameRate (Dataset): shape=(), dtype=int32
frameSignalLine (Dataset): shape=(2,), dtype=int32
fullFieldContrast (Dataset): shape=(3,), dtype=int32
galvoChannels (Dataset): shape=(3,), dtype=float64
gammaErrorPolicy (Dataset): sh

In [5]:
import h5py
import numpy as np

def inspect_hdf5_field(file_path, field_path, max_items=20):
    """
    Inspect a specific field/dataset in an HDF5 file.
    
    Args:
        file_path: Path to the HDF5 file
        field_path: Path to the field (e.g., 'group1/dataset_name')
        max_items: Maximum number of items to display for arrays
    """
    try:
        with h5py.File(file_path, 'r') as f:
            if field_path not in f:
                print(f"Field '{field_path}' not found in file.")
                print("Available fields:")
                f.visititems(lambda name, obj: print(f"  {name}") if isinstance(obj, h5py.Dataset) else None)
                return
            
            dataset = f[field_path]
            
            print(f"📄 Dataset: {field_path}")
            print("=" * 50)
            print(f"Shape: {dataset.shape}")
            print(f"Data type: {dataset.dtype}")
            print(f"Size: {dataset.size} elements")
            
            # Print attributes
            if dataset.attrs:
                print("\nAttributes:")
                for attr_name, attr_value in dataset.attrs.items():
                    print(f"  {attr_name}: {attr_value}")
            
            # Display data based on size and dimensionality
            print(f"\nData preview:")
            if dataset.size == 0:
                print("  (empty dataset)")
            elif dataset.size <= max_items:
                print(f"  Full data: {dataset[:]}")
            else:
                if dataset.ndim == 1:
                    print(f"  First {max_items} items: {dataset[:max_items]}")
                    if dataset.size > max_items:
                        print(f"  ... ({dataset.size - max_items} more items)")
                elif dataset.ndim == 2:
                    rows_to_show = min(5, dataset.shape[0])
                    cols_to_show = min(10, dataset.shape[1])
                    print(f"  First {rows_to_show}x{cols_to_show} elements:")
                    print(f"  {dataset[:rows_to_show, :cols_to_show]}")
                    if dataset.shape[0] > rows_to_show or dataset.shape[1] > cols_to_show:
                        print(f"  ... (shape: {dataset.shape})")
                else:
                    print(f"  First few elements: {dataset.flat[:max_items]}")
            
            # Basic statistics for numeric data
            if np.issubdtype(dataset.dtype, np.number) and dataset.size > 0:
                print(f"\nBasic statistics:")
                print(f"  Min: {np.min(dataset)}")
                print(f"  Max: {np.max(dataset)}")
                print(f"  Mean: {np.mean(dataset):.6f}")
                if dataset.size > 1:
                    print(f"  Std: {np.std(dataset):.6f}")
                    
    except Exception as e:
        print(f"Error inspecting field: {e}")

def list_all_datasets(file_path, filter_pattern=None):
    """
    List all datasets in the HDF5 file, optionally filtered by pattern.
    
    Args:
        file_path: Path to the HDF5 file
        filter_pattern: Optional string pattern to filter dataset names
    """
    datasets = []
    
    def collect_datasets(name, obj):
        if isinstance(obj, h5py.Dataset):
            if filter_pattern is None or filter_pattern.lower() in name.lower():
                datasets.append((name, obj.shape, obj.dtype))
    
    try:
        with h5py.File(file_path, 'r') as f:
            f.visititems(collect_datasets)
            
        print(f"📂 Datasets in {file_path}")
        if filter_pattern:
            print(f"   (filtered by: '{filter_pattern}')")
        print("=" * 70)
        
        for name, shape, dtype in sorted(datasets):
            print(f"{name:<40} {str(shape):<20} {dtype}")
            
    except Exception as e:
        print(f"Error listing datasets: {e}")

def inspect_multiple_fields(file_path, field_paths, max_items=10):
    """
    Inspect multiple fields at once for comparison.
    
    Args:
        file_path: Path to the HDF5 file
        field_paths: List of field paths to inspect
        max_items: Maximum items to show per field
    """
    for i, field_path in enumerate(field_paths):
        if i > 0:
            print("\n" + "="*80 + "\n")
        inspect_hdf5_field(file_path, field_path, max_items)

def search_datasets_by_attribute(file_path, attr_name, attr_value=None):
    """
    Find datasets that have a specific attribute.
    
    Args:
        file_path: Path to the HDF5 file
        attr_name: Name of the attribute to search for
        attr_value: Optional specific value to match (if None, just checks if attribute exists)
    """
    matches = []
    
    def check_attributes(name, obj):
        if isinstance(obj, h5py.Dataset):
            if attr_name in obj.attrs:
                if attr_value is None or obj.attrs[attr_name] == attr_value:
                    matches.append((name, obj.attrs[attr_name]))
    
    try:
        with h5py.File(file_path, 'r') as f:
            f.visititems(check_attributes)
            
        print(f"🔍 Datasets with attribute '{attr_name}':")
        if attr_value is not None:
            print(f"   (value = {attr_value})")
        print("=" * 50)
        
        for name, value in matches:
            print(f"{name}: {value}")
            
    except Exception as e:
        print(f"Error searching attributes: {e}")

# Usage examples:

# 1. Inspect a specific field
# inspect_hdf5_field('/path/to/file.h5', 'processing/ecephys/LFP/ElectricalSeries/data')

# 2. List all datasets
# list_all_datasets('/path/to/file.h5')

# 3. List datasets containing specific text
# list_all_datasets('/path/to/file.h5', filter_pattern='spike')

# 4. Inspect multiple fields
# inspect_multiple_fields('/path/to/file.h5', [
#     'acquisition/ElectricalSeries/data',
#     'processing/behavior/Position/SpatialSeries/data'
# ])

# 5. Search by attribute
# search_datasets_by_attribute('/path/to/file.h5', 'unit', 'volts')

In [12]:
filepath = '/Volumes/aind/scratch/andrew.shelton/NPUltra_data/raw_npultra_data/2024-05-14_714527/behavior/RFMapping_714527_20240514_104628.hdf5'

list_all_datasets(filepath, 'stimFrames')

📂 Datasets in /Volumes/aind/scratch/andrew.shelton/NPUltra_data/raw_npultra_data/2024-05-14_714527/behavior/RFMapping_714527_20240514_104628.hdf5
   (filtered by: 'stimFrames')
interStimFrames                          ()                   int32
stimFrames                               ()                   int32
